# Avaliação adversarial do novo dataset (wrapper fino)

Este notebook **não implementa** o protocolo adversarial. Ele chama a CLI do
pacote `new_data_eval`, a mesma usada em
`PYTHONPATH=code python -m new_data_eval adversarial`. O ataque, as máscaras, o
threshold calibrado, os checkpoints e os resultados vivem apenas no pacote
(`code/new_data_eval/`). Veja `docs/NEW_DATASET.md`.

O threshold adversarial é lido do resultado normal de FPR 10% no mesmo
`RESULTS_DIR`; rode o notebook normal (ou a CLI `normal` com o mesmo
`--results-dir`) antes. Este notebook grava em
`results/new_data/notebook_smoke/`, isolado do protocolo completo. Para o
protocolo completo, use `MAX_SAMPLES = 5000` e `N_MASKS = 100`.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start):
    """Return the repository root that contains ``code/new_data_eval``."""
    for candidate in (start, *start.parents):
        if (candidate / "code" / "new_data_eval").is_dir():
            return candidate
    raise RuntimeError("Could not find the repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "code"))

from new_data_eval import cli
from new_data_eval.data import ATTACK_LABELS

## Parâmetros de inspeção

`ATTACK` vem da lista canônica do pacote (nenhum nome de ataque duplicado
aqui); `VARIANT` seleciona o diretório de resultados. `MAX_SAMPLES` e
`N_MASKS` são apenas para smoke tests. Os caminhos de resultado são derivados
pelas funções do pacote.

In [ ]:
DATA_DIR = REPO_ROOT / "data" / "new_data"
MODEL_DIR = REPO_ROOT / "models" / "new_data_model"
RESULTS_DIR = Path(cli.DEFAULT_RESULTS_DIR) / "notebook_smoke"

# The default label is the last entry of ATTACK_LABELS.  Change it to
# any entry of the list to inspect another class.
ATTACK = ATTACK_LABELS[-1]
VARIANT = cli.ADVERSARIAL_VARIANTS[0]

# Smoke defaults: small and quick.  For the full protocol use
# MAX_SAMPLES = 5000 and N_MASKS = 100.
MAX_SAMPLES = 5
N_MASKS = 5
BATCH_SIZE = 2
SEED = 2019

## Rodar a avaliação adversarial

In [ ]:
adversarial_args = [
    "adversarial",
    "--data-dir", str(DATA_DIR),
    "--model-dir", str(MODEL_DIR),
    "--results-dir", str(RESULTS_DIR),
    "--attack", ATTACK,
    "--variant", VARIANT,
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--n-masks", str(N_MASKS),
]
if MAX_SAMPLES is not None:
    adversarial_args += ["--max-samples", str(MAX_SAMPLES)]
if cli.main(adversarial_args) != 0:
    raise RuntimeError("adversarial evaluation failed")

## Inspecionar o JSON gravado

In [ ]:
import json

result_file = cli.result_path(RESULTS_DIR, VARIANT, ATTACK)
payload = json.loads(result_file.read_text(encoding="utf-8"))
print(result_file)
for key in (
    "thresholds",
    "scoring",
    "counts",
    "fpr_at_attack_threshold",
    "normal_tpr_at_attack_threshold",
    "victim_tpr",
    "attack_success_rate",
    "conditional_drop",
    "fooled_rate",
):
    print(f"{key}: {payload[key]}")